In [ ]:
import pathlib

import sklearn.neighbors
# import networkx as nx
import igraph as ig
import leidenalg

import numpy as np
# import matplotlib
# import matplotlib.pyplot as plt
import pandas as pd

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import GraphRenderer, Ellipse, StaticLayoutProvider, HoverTool, MultiLine, Circle
from bokeh.palettes import Category20

import trapallada.plot
import trapallada.analysis
# import ipywidgets as widgets

In [ ]:
output_notebook()

So that `import`ed Python modules are automatically reloaded when modified

In [ ]:
%load_ext autoreload
%autoreload 2

# Input Files

Relative path and *Base* file name for the results (trained model). It should be a `str`.

In [ ]:
input_files_stem = '../results/epochs/20000/4_sources/epochs'
# input_files_stem = '../results/4_sources/vanilla_5000_epochs_05_x_sd'
# input_files_stem = '../results/4_sources/vanilla_5000_epochs' # rotated "C" below

It is turned into a `Pathlib`

In [ ]:
input_files_stem = pathlib.Path(input_files_stem)
input_files_stem

From the results directory,
- the `Dataset`,
- the headers of the corresponding files,
- the ML model,
- a `DataFrame` with results from training, 
- a `dict` with the parameters

In [ ]:
ds, positions_df, model, results_df, parameters, analysis_df = trapallada.analysis.load_results_dir(input_files_stem, analysis_also=True)

For the sake of convenience

In [ ]:
zs_columns = [f'z_{i}' for i in range(1, parameters['model']['dim_z']+1)]
zs_columns[:7]

The *adjacency* matrix

In [ ]:
adj_matrix = sklearn.neighbors.kneighbors_graph(analysis_df[zs_columns].values, n_neighbors=10, mode='distance')
adj_matrix.shape

In [ ]:
type(adj_matrix)

**Similarity** metric (as suggested by JCS)

In [ ]:
alpha = 1.
adj_matrix.data = np.exp(- alpha * adj_matrix.data)

In [ ]:
def similarity_matix_to_graph(M):
    matrix_coo = M.tocoo()
    edges = list(zip(matrix_coo.row, matrix_coo.col))
    weights = matrix_coo.data  # Distances are the weights

    g = ig.Graph(edges=edges, directed=False)
    g.es['weight'] = weights

    return g

G = similarity_matix_to_graph(adj_matrix)

Leidgen algorithm using [leidenalg](https://leidenalg.readthedocs.io/en/stable/index.html) implementation.

In [ ]:
partition = leidenalg.find_partition(G, leidenalg.ModularityVertexPartition)
partition

In [ ]:
communities = partition.membership
n_communities = max(communities) + 1
n_communities

In [ ]:
layout = G.layout('auto')
layout

In [ ]:
palette = Category20[20]
nodes_df = pd.DataFrame(
    {
        'community': communities,
        # 'fill_color': [palette[c ]for c in communities],
        'fill_color': [palette[c % 20]for c in communities],
        'coordinates': layout.coords
    },
    index=analysis_df.index
)
unknown_series = analysis_df['main_type'].isna()
nodes_df.loc[unknown_series, 'marker'] = 'square'
nodes_df.loc[~unknown_series, 'marker'] = 'diamond'
nodes_df

In [ ]:
edges_df = pd.DataFrame(
    [(edge.source, edge.target) for edge in G.es], columns=['start', 'end']
)
edges_df


In [ ]:
from bokeh.models import Scatter

def plot_graph(nodes_df, edges_df):

    fig = figure(
        title="Latent space",
        width=1_200, height=1024,
        # x_range=(-1.1,1.1), y_range=(-1.1,1.1),
        # tools="", toolbar_location=None
    )

    graph = GraphRenderer()

    # graph.node_renderer.glyph = Ellipse(height=6., width=6., fill_color="fill_color")
    # graph.node_renderer.glyph = Circle(radius=3., fill_color="fill_color")
    graph.node_renderer.glyph = Scatter(size=8, fill_color="fill_color", marker="marker")

    graph.node_renderer.data_source.data = nodes_df.reset_index()[['index', 'fill_color', 'marker']].to_dict(orient='list')

    graph.node_renderer.data_source.data['community'] = nodes_df['community'].to_list()
    graph.node_renderer.data_source.data['main_type'] = analysis_df['main_type'].to_list()
    hover_tool = HoverTool(
        tooltips=[
            ("Community", "@community"),
            ("main_type", "@main_type"),
            ("index", "@index"),
        ]
    )

    graph.edge_renderer.data_source.data = edges_df.to_dict('list')
    graph.edge_renderer.glyph = MultiLine(line_color="black", line_alpha=0.01)

    graph.layout_provider = StaticLayoutProvider(graph_layout=nodes_df['coordinates'].to_dict())

    fig.renderers.append(graph)
    fig.add_tools(hover_tool)

    # show(plot)
    return fig

In [ ]:
show(plot_graph(nodes_df, edges_df))

In [ ]:
# ig.plot(G, 'comm.png', layout=layout, vertex_color=node_colors, vertex_size=20, vertex_label=None)